# NeuralMath — 신경망의 수학적 해부

본 논문은 간단한 분류 문제를 통해 인공 신경망을 수학적으로 소개한다. 선형 분류기에서 인공 뉴런의 개념을 자연스럽게 도출한 뒤, 가중치, 편향, 활성화 함수, 그리고 여러 뉴런을 작은 네트워크로 결합하는 방법을 설명한다. 학습은 손실 함수, 도함수, 경사하강법, 역전파를 통해 소개한다. 마지막으로 모델을 Python으로 직접 구현하여 수식과 알고리즘의 대응을 명확히 보인다. 목적은 해석기하, 선형대수, 미적분에서 익숙한 개념들이 신경망의 기본 작동 원리를 이해하는 데 접근하기 쉬운 기초가 됨을 보여 주는 것이다.

**NeuralMath.org** · *A anatomia matemática de uma rede neural*

변수와 함수 이름은 영어로 유지하고, 설명문은 이 페이지의 언어로 제공합니다.


## 1. 작은 신경망 구성하기

변수와 함수 이름은 영어로 유지하고, 설명문은 이 페이지의 언어로 제공합니다.

$$\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad \mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),$$

$$z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad \widehat y=\sigma(z^{(2)}).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivative(s):
    return (s > 0).astype(float)


def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def base_boundary(m):
    return (
        6.68
        + 0.74 * np.exp(-((m - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(m - 202) / 5))
    )


## 2. 문제를 수학적으로 표현하기

신경망의 작동 방식을 이해하기 위해 의도적으로 단순한 문제, 즉 사과와 오렌지를 구별하는 문제에서 시작한다. 과일에는 색, 질량, 지름, 질감, 향처럼 분류에 도움이 될 수 있는 많은 특성이 있다. 그러나 첫 번째 수학 모델에 이 모든 복잡성을 포함할 필요는 없다. 여기서는 쉽게 측정할 수 있는 두 양만 사용한다. 질량은 그램, 지름은 센티미터 단위로 측정한다.

$$x_j^{\ast}=\frac{x_j-\mu_j}{\sigma_j}.$$


In [ ]:
def build_dataset():
    masses_1 = np.linspace(118, 212, 16)
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    apple_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    apple_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    orange_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    orange_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_train = np.vstack([apple_1, apple_2, orange_1, orange_2])
    y_train = np.array([1.0] * 32 + [0.0] * 32)

    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_test = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = build_dataset()
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
Xn = (X_train - mean) / std

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('mean:', mean)
print('std:', std)


## 3. 신경망은 어떻게 학습하는가?

가중치와 편향은 미리 알 수 없다. 정답이 알려진 예제를 사용해 조정해야 한다. 이 과정을 학습 이라 하며 다음 순환으로 요약할 수 있다. \[\boxed{\text{예측}}
\longrightarrow
\boxed{\text{오차 측정}}
\longrightarrow
\boxed{\text{매개변수 조정}}
\longrightarrow
\boxed{\text{반복}}.\]


In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * rng.standard_normal(12)
b2 = 0.0

learning_rate = 0.05
n_epochs = 6000
history = []

for epoch in range(n_epochs):
    for i in rng.permutation(len(Xn)):
        x = Xn[i]
        target = y_train[i]

        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        delta2 = (y_hat - target) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivative(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    errors = []
    for x, target in zip(Xn, y_train):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        errors.append(0.5 * (target - y_hat) ** 2)
    history.append(float(np.mean(errors)))

print(f"E_final: {history[-1]:.10f}")


## 4. 수학에서 Python으로

이제 수식을 실행 가능한 프로그램으로 옮긴다. 목적은 전용 신경망 라이브러리를 사용하는 것이 아니라, 본 논문에서 전개한 수학 연산을 NumPy 로 직접 구현하는 것이다 . 이를 통해 선형 결합, 활성화 함수, 손실 계산, 역전파, 경사하강법이 코드에 어떻게 나타나는지 한 줄씩 확인할 수 있다.


In [ ]:
def predict(X):
    X_scaled = (X - mean) / std
    outputs = []
    for x in X_scaled:
        h = relu(W1 @ x + b1)
        outputs.append(sigmoid(W2 @ h + b2))
    return np.array(outputs)


pred_train = predict(X_train)
pred_test = predict(X_test)

print(f"acc_train: {np.mean((pred_train >= 0.5) == y_train):.4f}")
print(f"acc_test: {np.mean((pred_test >= 0.5) == y_test):.4f}")
print('y_hat_test:', np.round(pred_test, 6))

masses = np.linspace(116, 214, 500)
diameters = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(masses, diameters)
grid = np.column_stack([M.ravel(), D.ravel()])
P = predict(grid).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mask = y_train == 1.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='사과 — 학습')
mask = y_train == 0.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='오렌지 — 학습')
mask = y_test == 1.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^', s=92,
           color='#1f77b4', label='사과 — 테스트')
mask = y_test == 0.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='오렌지 — 테스트')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('질량 (g)')
ax.set_ylabel('지름 (cm)')
ax.set_title('학습 후 신경망의 출력', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. NeuralMath

본 논문과 관련된 모든 계산 자료 — 코드와 실행 가능한 Google Colab 노트북을 포함하여 — 는 https://neuralmath.org 에서 공개한다. 독자는 여기에서 예제를 조사하고 재현하며 수정하는 데 필요한 자료에 한 번에 접근할 수 있다.
